# 03 — Sales by Product Category
Sales distribution and absolute revenue/profit across Furniture, Office Supplies, and Technology.


In [ ]:
import os, sys
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

os.environ['JAVA_HOME']             = '/usr/local/java'
os.environ['SPARK_HOME']            = '/usr/local/spark'
os.environ['HADOOP_CONF_DIR']       = '/usr/local/hadoop/etc/hadoop'
os.environ['PYSPARK_PYTHON']        = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

try:
    spark.stop()
except:
    pass

# Local mode — no Hive/hive-metastore dependency
spark = (SparkSession.builder
    .appName("Superstore Analytics")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# ── Load CSVs and register temp views ─────────────────────────────────────────
DATA = "/usr/local/hadoop/etc/hadoop/assessment-2"

def load(filename, renames):
    df = spark.read.option("header", "true").option("inferSchema", "true") \
             .csv(f"{DATA}/{filename}")
    for old, new in renames.items():
        df = df.withColumnRenamed(old, new)
    return df

customers  = load("customers.csv",  {"Customer ID": "customer_id",
                                      "Customer Name": "customer_name",
                                      "Segment": "segment"})
orders     = load("orders.csv",     {"Order ID": "order_id",
                                      "Order Date": "order_date",
                                      "Ship Date": "ship_date",
                                      "Ship Mode": "ship_mode",
                                      "Customer ID": "customer_id",
                                      "Postal Code": "postal_code"})
order_items = load("order_items.csv", {"Row ID": "row_id",
                                        "Order ID": "order_id",
                                        "Product ID": "product_id",
                                        "Sales": "sales",
                                        "Quantity": "quantity",
                                        "Discount": "discount",
                                        "Profit": "profit"})
products   = load("products.csv",   {"Product ID": "product_id",
                                      "Product Name": "product_name",
                                      "Category": "category",
                                      "Sub-Category": "sub_category"})
locations  = load("locations.csv",  {"Postal Code": "postal_code",
                                      "City": "city",
                                      "State": "state",
                                      "Country": "country",
                                      "Region": "region"})

customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")
products.createOrReplaceTempView("products")
locations.createOrReplaceTempView("locations")

print("Spark", spark.version, "ready — all tables loaded.")
spark.sql("SHOW TABLES").show()

# ── Global chart style ─────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = sns.color_palette("muted")

def fmt_usd(v):
    if abs(v) >= 1_000_000:
        return f"${v/1_000_000:.2f}M"
    if abs(v) >= 1_000:
        return f"${v/1_000:.1f}K"
    return f"${v:.0f}"


In [ ]:
# ── Sales Distribution by Category ───────────────────────────────────────────
cat = spark.sql("""
    SELECT p.category,
           ROUND(SUM(oi.sales),  2) AS sales,
           ROUND(SUM(oi.profit), 2) AS profit
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.category
    ORDER BY sales DESC
""").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Sales Distribution by Product Category",
             fontsize=14, fontweight="bold", y=1.02)

wedges, texts, autotexts = axes[0].pie(
    cat["sales"], labels=cat["category"],
    autopct="%1.1f%%", colors=PALETTE[:len(cat)],
    startangle=140,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    pctdistance=0.78,
)
for at in autotexts:
    at.set_fontsize(11); at.set_fontweight("bold")
axes[0].set_title("Sales Share")

bar_w = 0.35
x     = range(len(cat))
b1 = axes[1].bar([i - bar_w/2 for i in x], cat["sales"],
                 bar_w, color=PALETTE[0], label="Sales",  edgecolor="white")
b2 = axes[1].bar([i + bar_w/2 for i in x], cat["profit"],
                 bar_w, color=PALETTE[1], label="Profit", edgecolor="white")
for bar in b1:
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 500,
                 fmt_usd(bar.get_height()),
                 ha="center", va="bottom", fontsize=8, color=PALETTE[0])
for bar in b2:
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 200,
                 fmt_usd(bar.get_height()),
                 ha="center", va="bottom", fontsize=8, color=PALETTE[1])
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(cat["category"], fontsize=10)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_usd(v)))
axes[1].legend(frameon=False)
axes[1].set_title("Sales & Profit (USD)")
axes[1].grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
spark.stop()
print("Spark stopped.")
